# MATH 5320 — Demo: VaR/ES & Regulatory Capital

**Table of Contents**

1. [Case 1: Parametric VaR/ES](#case1) — Delta-normal (rolling window) and EWMA estimates for a 100 AAPL + 50 CAT portfolio, Bloomberg CSV data, pricing date 2026-02-11.
2. [Case 2: HW10 RWA & Capital](#case2) — Risk-weighted assets and Basel III CET1 capital ratio for a stylised bank balance sheet.

In [ ]:
import sys
import numpy as np
import pandas as pd
from scipy.stats import norm
import matplotlib.pyplot as plt

sys.path.insert(0, '..')

from src.risk.regulatory import risk_weighted_assets, capital_ratio
from src.credit.cds import cds_par_spread_constant_hazard

## Case 1 — Parametric VaR & ES
<a id='case1'></a>

For a portfolio with daily mean $\mu_p$ and daily volatility $\sigma_p$, the **delta-normal** (parametric) measures over horizon $h$ are:

$$\text{VaR}_\alpha = -\left(\mu_p h - z_\alpha \sigma_p \sqrt{h}\right) \times V$$

$$\text{ES}_\alpha = -\left(\mu_p h - \frac{\varphi(z_\alpha)}{1-\alpha} \sigma_p \sqrt{h}\right) \times V$$

where $z_\alpha = \Phi^{-1}(\alpha)$ is the standard-normal quantile and $\varphi$ is the standard-normal PDF.

**Portfolio:** 100 shares AAPL + 50 shares CAT.  
**Pricing date:** 2026-02-11.  
**Estimation window:** trailing 252 trading days.

In [ ]:
def load_bbg(path, ticker):
    df = pd.read_csv(path, parse_dates=['Dates'])
    df = df.rename(columns={'Dates': 'Date'})[['Date', 'PX_LAST']].dropna()
    df = df[~df['Date'].duplicated(keep='last')].set_index('Date').sort_index()
    return df['PX_LAST'].rename(ticker)

aapl = load_bbg('../data/AAPL-bloomberg.csv', 'AAPL')
cat  = load_bbg('../data/CAT-bloomberg.csv',  'CAT')
prices = pd.concat([aapl, cat], axis=1).dropna()
print(f"Loaded {len(prices):,} trading days, last: {prices.index[-1].date()}")
print(prices.tail(3))

In [ ]:
n_aapl, n_cat = 100, 50
last    = prices.iloc[-1]
V_aapl  = n_aapl * last['AAPL']
V_cat   = n_cat  * last['CAT']
V       = V_aapl + V_cat

# Trailing 252-day log-returns
ret     = np.log(prices / prices.shift(1)).dropna().iloc[-252:]
mu_d    = ret.mean().values        # daily mean vector
cov_d   = ret.cov().values         # daily covariance matrix
wgt     = np.array([V_aapl / V, V_cat / V])
port_mu = wgt @ mu_d
port_sig= np.sqrt(wgt @ cov_d @ wgt)

conf    = 0.99
h       = 1                         # 1-day horizon
z_alpha = norm.ppf(conf)
VaR_1d  = (-port_mu * h + z_alpha * port_sig * np.sqrt(h)) * V
ES_1d   = (-port_mu * h + norm.pdf(z_alpha) / 0.01 * port_sig * np.sqrt(h)) * V
VaR_10d = (-port_mu * 10 + z_alpha * port_sig * np.sqrt(10)) * V  # sqrt-T scaling

print(f"Portfolio:  100 AAPL @ ${last['AAPL']:.2f} + 50 CAT @ ${last['CAT']:.2f}")
print(f"Value:      ${V:,.2f}  (AAPL: ${V_aapl:,.2f}  |  CAT: ${V_cat:,.2f})")
print(f"Daily \u03c3:    AAPL={np.sqrt(cov_d[0,0]):.4f}  |  CAT={np.sqrt(cov_d[1,1]):.4f}")
print(f"Port \u03c3_d:   {port_sig:.4f}  |  z_0.99 = {z_alpha:.4f}")
print()
print(f"Parametric VaR (99%, 1d):  ${VaR_1d:,.2f}")
print(f"Parametric ES  (99%, 1d):  ${ES_1d:,.2f}")
print(f"Parametric VaR (99%, 10d, \u221aT): ${VaR_10d:,.2f}")

In [ ]:
lam  = 0.94
T    = len(ret)
W    = np.array([(1 - lam) * lam**i for i in range(T)])[::-1]
W   /= W.sum()
mu_e = (ret.values * W[:, None]).sum(axis=0)
cov_e = sum(
    W[t] * np.outer(ret.values[t] - mu_e, ret.values[t] - mu_e)
    for t in range(T)
)
sig_e = np.sqrt(wgt @ cov_e @ wgt)
VaR_e = (-wgt @ mu_e + z_alpha * sig_e) * V
ES_e  = (-wgt @ mu_e + norm.pdf(z_alpha) / 0.01 * sig_e) * V
print(f"EWMA (\u03bb=0.94) VaR 99% 1d: ${VaR_e:,.2f}")
print(f"EWMA (\u03bb=0.94) ES  99% 1d: ${ES_e:,.2f}")

### Interpretation

- **Rolling window vs EWMA:** The equal-weighted 252-day window treats all historical observations identically, so large drawdowns from the estimation window (e.g., 2022 volatility spikes) receive full weight. EWMA with $\lambda = 0.94$ exponentially down-weights older observations, making it more responsive to recent calm periods — hence EWMA VaR is typically lower when recent volatility has subsided.

- **ES ≥ VaR (coherence):** Expected Shortfall is always at least as large as VaR at the same confidence level. This follows directly from the formulas: $\varphi(z_\alpha)/(1-\alpha) > z_\alpha$ for any $\alpha \in (0, 1)$, so the ES coefficient on $\sigma_p$ exceeds the VaR coefficient. ES is a coherent risk measure (satisfies sub-additivity); VaR is not.

## Case 2 — HW10: Risk-Weighted Assets & Capital Ratio
<a id='case2'></a>

Basel III requires banks to maintain a minimum **CET1 capital ratio** of 8% of risk-weighted assets:

$$\text{RWA} = \sum_i w_i A_i$$

$$k = \frac{\text{Equity}}{\text{RWA}} \geq 8\%$$

**Balance sheet:**

| Asset | Amount | Risk weight |
|---|---|---|
| Cash | \$69,000 | 0.00 |
| Mortgages | \$73,000 | 0.45 |
| Corporate loans | \$47,000 | 1.00 |
| **Total** | **\$189,000** | |

Deposits = \$182,000 → Equity = \$7,000

In [ ]:
assets  = [69_000, 73_000, 47_000]
weights = [0.00,   0.45,   1.00]
deposits = 182_000

total_assets = sum(assets)
equity       = total_assets - deposits
rwa          = risk_weighted_assets(assets, weights)
cap          = capital_ratio(equity=equity, rwa=rwa)

print("HW10 \u2014 Bank Balance Sheet")
print(f"  Cash       : ${assets[0]:>8,}  \u00d7  {weights[0]:.2f}  \u2192  RWA contrib = ${assets[0]*weights[0]:>8,.0f}")
print(f"  Mortgages  : ${assets[1]:>8,}  \u00d7  {weights[1]:.2f}  \u2192  RWA contrib = ${assets[1]*weights[1]:>8,.0f}")
print(f"  Corp Loans : ${assets[2]:>8,}  \u00d7  {weights[2]:.2f}  \u2192  RWA contrib = ${assets[2]*weights[2]:>8,.0f}")
print(f"  {'\u2500'*55}")
print(f"  Total assets: ${total_assets:,}  |  Deposits: ${deposits:,}")
print(f"  Equity      : ${equity:,}  (= assets \u2212 liabilities)")
print(f"  RWA         : ${rwa:,.0f}")
print()
print(f"  Capital ratio : {cap['ratio']:.4%}  (floor = {cap['floor']:.0%})")
print(f"  Basel III test: {'PASS \u2713' if cap['pass'] else 'FAIL \u2717'}")
print()
print(f"  HW10 expected: RWA = 79,850  |  ratio = 8.77%  |  PASS")

### Interpretation

The **8% CET1 floor** is the Basel III minimum for Common Equity Tier 1 capital. This stylised bank holds \$7,000 in equity against \$79,850 of RWA, yielding a capital ratio of **8.77%** — it barely passes the minimum requirement.

Key observations:
- **Cash** carries a 0% risk weight under Basel III standardised approach, so holding more cash does not increase RWA.
- **Residential mortgages** receive a 45% weight (preferential treatment vs unsecured lending).
- **Corporate loans** attract a 100% weight — every dollar of corporate exposure translates directly into one dollar of RWA.
- A thin equity cushion (\$7k vs \$189k total assets) means the bank's **leverage ratio** (~3.7%) is also low; regulators apply both the RWA-based and leverage-based floors in practice.